# **PREPARE ACDC DATASET**
*   For YOLOv11 + Faster R-CNN / Detectron2
*   Split: 60% train / 20% val / 20% test
*   Weather: fog, rain, snow
*   Classes: person, bicycle, car, motorcycle, bus, truck


## **Import Librairies**

In [1]:
from pathlib import Path
import shutil
import random
import json
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

import cv2
import numpy as np
from PIL import Image

## **Mount Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Path configuration**

In [3]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Dissertation")

ACDC_RAW_ROOT = PROJECT_ROOT / "Datasets/raw/ACDC"
ACDC_RGB_ROOT = ACDC_RAW_ROOT / "rgb_anon"
ACDC_GT_ROOT = ACDC_RAW_ROOT / "gt_detection"

PROCESSED_ROOT = PROJECT_ROOT / "Datasets/processed"

ACDC_YOLO_ROOT = PROCESSED_ROOT / "acdc_yolo"
ACDC_VOC_ROOT = PROCESSED_ROOT / "acdc_voc"

## **Structure folders check**

In [4]:
assert ACDC_RGB_ROOT.exists(), f"Missing RGB folder: {ACDC_RGB_ROOT}"
assert ACDC_GT_ROOT.exists(), f"Missing GT detection folder: {ACDC_GT_ROOT}"

print("ACDC folders found.")
WEATHERS = ["fog", "rain", "snow"]
for weather in WEATHERS:
    print(f"\nWeather: {weather}")

    for split in ["train", "val"]:
        img_dir = ACDC_RGB_ROOT / weather / split
        json_path = ACDC_GT_ROOT / weather / f"instancesonly_{weather}_{split}_gt_detection.json"

        print(f"  {split}")
        print("    image folder exists:", img_dir.exists())
        print("    json exists:", json_path.exists())

        if img_dir.exists():
            print("    images:", len(list(img_dir.rglob("*_rgb_anon.png"))))

ACDC folders found.

Weather: fog
  train
    image folder exists: True
    json exists: True
    images: 400
  val
    image folder exists: True
    json exists: True
    images: 100

Weather: rain
  train
    image folder exists: True
    json exists: True
    images: 400
  val
    image folder exists: True
    json exists: True
    images: 100

Weather: snow
  train
    image folder exists: True
    json exists: True
    images: 400
  val
    image folder exists: True
    json exists: True
    images: 100


## **Load COCO annotations**

In [ ]:
CLASS_NAMES = ("person", "bicycle", "car", "motorcycle", "bus", "truck")

samples = []
global_category_lookup = {}

for weather in WEATHERS:
    for original_split in ["train", "val"]:

        json_path = ACDC_GT_ROOT / weather / f"instancesonly_{weather}_{original_split}_gt_detection.json"

        print("Loading:", json_path)

        with open(json_path, "r") as f:
            coco = json.load(f)

        category_lookup = {
            cat["id"]: cat["name"]
            for cat in coco["categories"]
        }

        global_category_lookup.update(category_lookup)

        annotations_per_image = defaultdict(list)

        for ann in coco["annotations"]:
            cls_name = category_lookup[ann["category_id"]]

            if cls_name in CLASS_NAMES:
                annotations_per_image[ann["image_id"]].append(ann)

        for image_info in coco["images"]:
            image_id = image_info["id"]
            relative_path = image_info["file_name"]

            image_path = ACDC_RGB_ROOT / relative_path

            if not image_path.exists():
                print("Missing image:", image_path)
                continue

            samples.append({
                "image_id": image_id,
                "weather": weather,
                "original_split": original_split,
                "image_path": image_path,
                "width": image_info["width"],
                "height": image_info["height"],
                "annotations": annotations_per_image[image_id],
            })

print("\nTotal matched samples:", len(samples))
print("Samples per weather:", Counter(s["weather"] for s in samples))
print("Samples per original split:", Counter(s["original_split"] for s in samples))

Loading: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/raw/ACDC/gt_detection/fog/instancesonly_fog_train_gt_detection.json
Loading: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/raw/ACDC/gt_detection/fog/instancesonly_fog_val_gt_detection.json
Loading: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/raw/ACDC/gt_detection/rain/instancesonly_rain_train_gt_detection.json
Loading: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/raw/ACDC/gt_detection/rain/instancesonly_rain_val_gt_detection.json
Loading: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/raw/ACDC/gt_detection/snow/instancesonly_snow_train_gt_detection.json
Loading: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/raw/ACDC/gt_detection/snow/instancesonly_snow_val_gt_detection.json

Total matched samples: 2700
Samples per weather: Counter({'rain': 1700, 'fog': 500, 'snow': 500})
Samples per original split: Counter({'train': 2400, 'val': 300})


## **Check classes**

In [ ]:
print("Categories found in ACDC JSONs:")

for cat_id, cat_name in sorted(global_category_lookup.items()):
    print(cat_id, ":", cat_name)

Categories found in ACDC JSONs:
24 : person
25 : rider
26 : car
27 : truck
28 : bus
31 : train
32 : motorcycle
33 : bicycle


## **Check object distribution**

In [ ]:
class_counter = Counter()
weather_class_counter = defaultdict(Counter)
empty_images = 0

for sample in samples:
    if len(sample["annotations"]) == 0:
        empty_images += 1

    for ann in sample["annotations"]:
        cls_name = global_category_lookup[ann["category_id"]]

        if cls_name in CLASS_NAMES:
            class_counter[cls_name] += 1
            weather_class_counter[sample["weather"]][cls_name] += 1

print("Global class distribution:")
for cls in CLASS_NAMES:
    print(f"{cls}: {class_counter[cls]}")

print("\nClass distribution per weather:")
for weather in WEATHERS:
    print(f"\n{weather}")
    for cls in CLASS_NAMES:
        print(f"  {cls}: {weather_class_counter[weather][cls]}")

print("\nImages without selected objects:", empty_images)
print("Total images:", len(samples))

Global class distribution:
person: 2842
bicycle: 569
car: 11702
motorcycle: 335
bus: 291
truck: 1127

Class distribution per weather:

fog
  person: 238
  bicycle: 48
  car: 2134
  motorcycle: 25
  bus: 23
  truck: 401

rain
  person: 1826
  bicycle: 402
  car: 7197
  motorcycle: 262
  bus: 200
  truck: 553

snow
  person: 778
  bicycle: 119
  car: 2371
  motorcycle: 48
  bus: 68
  truck: 173

Images without selected objects: 41
Total images: 2700


## **Split 60/20/20**

In [ ]:
RANDOM_SEED = 42

In [ ]:
import random
random.seed(RANDOM_SEED)

In [ ]:
import numpy as np
np.random.seed(RANDOM_SEED)

In [ ]:
split_samples = {
    "train": [],
    "val": [],
    "test": []
}

for weather in WEATHERS:
    weather_samples = [s for s in samples if s["weather"] == weather]
    random.shuffle(weather_samples)

    n = len(weather_samples)
    n_train = int(0.60 * n)
    n_val = int(0.20 * n)

    train_part = weather_samples[:n_train]
    val_part = weather_samples[n_train:n_train + n_val]
    test_part = weather_samples[n_train + n_val:]

    split_samples["train"].extend(train_part)
    split_samples["val"].extend(val_part)
    split_samples["test"].extend(test_part)

print("Split sizes:")

for split, items in split_samples.items():
    print(split, len(items), Counter(s["weather"] for s in items))

Split sizes:
train 1620 Counter({'rain': 1020, 'fog': 300, 'snow': 300})
val 540 Counter({'rain': 340, 'fog': 100, 'snow': 100})
test 540 Counter({'rain': 340, 'fog': 100, 'snow': 100})


## **Output folders**

In [ ]:
for path in [ACDC_YOLO_ROOT, ACDC_VOC_ROOT]:
    if path.exists():
        print("Deleting:", path)
        shutil.rmtree(path)

for split in ["train", "val", "test", "test_fog", "test_rain", "test_snow"]:
    (ACDC_YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (ACDC_YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

(ACDC_VOC_ROOT / "JPEGImages").mkdir(parents=True, exist_ok=True)
(ACDC_VOC_ROOT / "Annotations").mkdir(parents=True, exist_ok=True)
(ACDC_VOC_ROOT / "ImageSets/Main").mkdir(parents=True, exist_ok=True)

print("Output folders ready.")

Deleting: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/acdc_yolo
Deleting: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/acdc_voc
Output folders ready.


## **YOLO label**

In [ ]:
def write_yolo_label(label_path, annotations, width, height):
    lines = []

    for ann in annotations:
        cls_name = global_category_lookup[ann["category_id"]]

        if cls_name not in CLASS_TO_ID:
            continue

        x, y, w, h = ann["bbox"]

        x_center = (x + w / 2) / width
        y_center = (y + h / 2) / height
        norm_w = w / width
        norm_h = h / height

        cls_id = CLASS_TO_ID[cls_name]

        lines.append(
            f"{cls_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
        )

    label_path.write_text("\n".join(lines))

## **VOC XML**

In [ ]:
def write_voc_xml(xml_path, image_filename, width, height, annotations):
    annotation = ET.Element("annotation")

    folder = ET.SubElement(annotation, "folder")
    folder.text = "JPEGImages"

    filename = ET.SubElement(annotation, "filename")
    filename.text = image_filename

    size = ET.SubElement(annotation, "size")

    ET.SubElement(size, "width").text = str(width)
    ET.SubElement(size, "height").text = str(height)
    ET.SubElement(size, "depth").text = "3"

    segmented = ET.SubElement(annotation, "segmented")
    segmented.text = "0"

    for ann in annotations:
        cls_name = global_category_lookup[ann["category_id"]]

        if cls_name not in CLASS_TO_ID:
            continue

        x, y, w, h = ann["bbox"]

        xmin = int(round(x))
        ymin = int(round(y))
        xmax = int(round(x + w))
        ymax = int(round(y + h))

        obj = ET.SubElement(annotation, "object")

        ET.SubElement(obj, "name").text = cls_name
        ET.SubElement(obj, "pose").text = "Unspecified"
        ET.SubElement(obj, "truncated").text = "0"
        ET.SubElement(obj, "difficult").text = "0"

        bndbox = ET.SubElement(obj, "bndbox")

        ET.SubElement(bndbox, "xmin").text = str(xmin)
        ET.SubElement(bndbox, "ymin").text = str(ymin)
        ET.SubElement(bndbox, "xmax").text = str(xmax)
        ET.SubElement(bndbox, "ymax").text = str(ymax)

    tree = ET.ElementTree(annotation)
    tree.write(xml_path)

## **Export YOLO & VOC (for use)**

In [ ]:
CLASS_NAMES = [
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "bus",
    "truck"
]

CLASS_TO_ID = {
    name: idx
    for idx, name in enumerate(CLASS_NAMES)
}

print(CLASS_TO_ID)

{'person': 0, 'bicycle': 1, 'car': 2, 'motorcycle': 3, 'bus': 4, 'truck': 5}


In [ ]:
all_split_ids = defaultdict(list)

export_stats = {
    "images": Counter(),
    "objects": Counter(),
    "objects_per_split": defaultdict(Counter),
    "objects_per_weather": defaultdict(Counter),
    "empty_images": Counter(),
}

for split_name, items in split_samples.items():

    print(f"\nExporting {split_name}: {len(items)} images")

    for sample in items:

        image_path = sample["image_path"]
        weather = sample["weather"]
        width = sample["width"]
        height = sample["height"]
        annotations = sample["annotations"]

        image_id = f"{weather}_{image_path.stem.replace('_rgb_anon', '')}"
        image_filename = f"{image_id}.jpg"

        image = Image.open(image_path).convert("RGB")

        # YOLO export
        yolo_img_path = ACDC_YOLO_ROOT / "images" / split_name / image_filename
        yolo_label_path = ACDC_YOLO_ROOT / "labels" / split_name / f"{image_id}.txt"

        image.save(yolo_img_path, quality=95)
        write_yolo_label(yolo_label_path, annotations, width, height)

        # Weather-specific YOLO test folders
        if split_name == "test":
            weather_split = f"test_{weather}"

            weather_img_path = ACDC_YOLO_ROOT / "images" / weather_split / image_filename
            weather_label_path = ACDC_YOLO_ROOT / "labels" / weather_split / f"{image_id}.txt"

            image.save(weather_img_path, quality=95)
            write_yolo_label(weather_label_path, annotations, width, height)

            all_split_ids[weather_split].append(image_id)

        # VOC export
        voc_img_path = ACDC_VOC_ROOT / "JPEGImages" / image_filename
        voc_xml_path = ACDC_VOC_ROOT / "Annotations" / f"{image_id}.xml"

        image.save(voc_img_path, quality=95)
        write_voc_xml(voc_xml_path, image_filename, width, height, annotations)

        all_split_ids[split_name].append(image_id)

        # Stats
        export_stats["images"][split_name] += 1

        if len(annotations) == 0:
            export_stats["empty_images"][split_name] += 1

        for ann in annotations:
            cls_name = global_category_lookup[ann["category_id"]]

            if cls_name in CLASS_NAMES:
                export_stats["objects"][cls_name] += 1
                export_stats["objects_per_split"][split_name][cls_name] += 1
                export_stats["objects_per_weather"][weather][cls_name] += 1

print("\nExport completed.")


Exporting train: 1620 images

Exporting val: 540 images

Exporting test: 540 images

Export completed.


## **Create VOC ImageSets**

In [ ]:
for split_name, ids in all_split_ids.items():
    split_file = ACDC_VOC_ROOT / "ImageSets/Main" / f"{split_name}.txt"
    split_file.write_text("\n".join(ids))

print("VOC ImageSets created:")

for file in sorted((ACDC_VOC_ROOT / "ImageSets/Main").glob("*.txt")):
    print(file.name, ":", len(file.read_text().splitlines()))

VOC ImageSets created:
test.txt : 540
test_fog.txt : 100
test_rain.txt : 340
test_snow.txt : 100
train.txt : 1620
val.txt : 540


## **Create YOLO YAML files**

In [ ]:
names_str = ", ".join([f"'{name}'" for name in CLASS_NAMES])

def write_yaml(path, text):
    path.write_text(text.strip())
    print("Created:", path)

global_yaml = f"""
path: {ACDC_YOLO_ROOT}
train: images/train
val: images/val
test: images/test

nc: {len(CLASS_NAMES)}
names: [{names_str}]
"""

fog_yaml = f"""
path: {ACDC_YOLO_ROOT}
train: images/train
val: images/val
test: images/test_fog

nc: {len(CLASS_NAMES)}
names: [{names_str}]
"""

rain_yaml = f"""
path: {ACDC_YOLO_ROOT}
train: images/train
val: images/val
test: images/test_rain

nc: {len(CLASS_NAMES)}
names: [{names_str}]
"""

snow_yaml = f"""
path: {ACDC_YOLO_ROOT}
train: images/train
val: images/val
test: images/test_snow

nc: {len(CLASS_NAMES)}
names: [{names_str}]
"""

write_yaml(ACDC_YOLO_ROOT / "acdc.yaml", global_yaml)
write_yaml(ACDC_YOLO_ROOT / "fog_only.yaml", fog_yaml)
write_yaml(ACDC_YOLO_ROOT / "rain_only.yaml", rain_yaml)
write_yaml(ACDC_YOLO_ROOT / "snow_only.yaml", snow_yaml)

Created: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/acdc_yolo/acdc.yaml
Created: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/acdc_yolo/fog_only.yaml
Created: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/acdc_yolo/rain_only.yaml
Created: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/acdc_yolo/snow_only.yaml


## **Verification**

In [ ]:
print("YOLO folders:")

for split in ["train", "val", "test", "test_fog", "test_rain", "test_snow"]:
    img_count = len(list((ACDC_YOLO_ROOT / "images" / split).glob("*.jpg")))
    label_count = len(list((ACDC_YOLO_ROOT / "labels" / split).glob("*.txt")))

    print(f"{split}: images={img_count}, labels={label_count}")

print("\nVOC folders:")
print("JPEGImages:", len(list((ACDC_VOC_ROOT / "JPEGImages").glob("*.jpg"))))
print("Annotations:", len(list((ACDC_VOC_ROOT / "Annotations").glob("*.xml"))))

print("\nVOC ImageSets:")

for split in ["train", "val", "test", "test_fog", "test_rain", "test_snow"]:
    split_file = ACDC_VOC_ROOT / "ImageSets/Main" / f"{split}.txt"

    if split_file.exists():
        print(split, ":", len(split_file.read_text().splitlines()))
    else:
        print(split, ": missing")

print("\nGlobal object distribution:")

for cls in CLASS_NAMES:
    print(f"{cls}: {export_stats['objects'][cls]}")

print("\nObject distribution per split:")

for split in ["train", "val", "test"]:
    print(f"\n{split}")

    for cls in CLASS_NAMES:
        print(f"  {cls}: {export_stats['objects_per_split'][split][cls]}")

print("\nObject distribution per weather:")

for weather in WEATHERS:
    print(f"\n{weather}")

    for cls in CLASS_NAMES:
        print(f"  {cls}: {export_stats['objects_per_weather'][weather][cls]}")

print("\nEmpty images per split:")

for split in ["train", "val", "test"]:
    print(split, ":", export_stats["empty_images"][split])

YOLO folders:
train: images=1620, labels=1620
val: images=540, labels=540
test: images=540, labels=540
test_fog: images=100, labels=100
test_rain: images=340, labels=340
test_snow: images=100, labels=100

VOC folders:
JPEGImages: 2700
Annotations: 2700

VOC ImageSets:
train : 1620
val : 540
test : 540
test_fog : 100
test_rain : 340
test_snow : 100

Global object distribution:
person: 2842
bicycle: 569
car: 11702
motorcycle: 335
bus: 291
truck: 1127

Object distribution per split:

train
  person: 1731
  bicycle: 337
  car: 7071
  motorcycle: 182
  bus: 190
  truck: 664

val
  person: 555
  bicycle: 115
  car: 2289
  motorcycle: 76
  bus: 52
  truck: 246

test
  person: 556
  bicycle: 117
  car: 2342
  motorcycle: 77
  bus: 49
  truck: 217

Object distribution per weather:

fog
  person: 238
  bicycle: 48
  car: 2134
  motorcycle: 25
  bus: 23
  truck: 401

rain
  person: 1826
  bicycle: 402
  car: 7197
  motorcycle: 262
  bus: 200
  truck: 553

snow
  person: 778
  bicycle: 119
  car: 

## **Save preparation summary**

In [ ]:
summary = {
    "dataset": "ACDC",
    "source_annotations": "gt_detection COCO JSON",
    "weather_conditions": WEATHERS,
    "classes": CLASS_NAMES,
    "random_seed": RANDOM_SEED,
    "split_ratio": {
        "train": 0.60,
        "val": 0.20,
        "test": 0.20
    },
    "raw_rgb_root": str(ACDC_RGB_ROOT),
    "raw_gt_root": str(ACDC_GT_ROOT),
    "yolo_output": str(ACDC_YOLO_ROOT),
    "voc_output": str(ACDC_VOC_ROOT),
    "split_sizes": {
        split: len(items)
        for split, items in split_samples.items()
    },
    "split_weather_distribution": {
        split: dict(Counter(s["weather"] for s in items))
        for split, items in split_samples.items()
    },
    "class_distribution": dict(export_stats["objects"]),
    "objects_per_split": {
        split: dict(export_stats["objects_per_split"][split])
        for split in ["train", "val", "test"]
    },
    "objects_per_weather": {
        weather: dict(export_stats["objects_per_weather"][weather])
        for weather in WEATHERS
    },
    "empty_images_per_split": dict(export_stats["empty_images"])
}

summary_path = PROCESSED_ROOT / "acdc_preparation_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("Saved summary:", summary_path)

Saved summary: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/acdc_preparation_summary.json


In [ ]:
ACDC_YAML = "/content/drive/MyDrive/Dissertation/Datasets/processed/acdc_yolo/acdc.yaml"
ACDC_VOC_ROOT = "/content/drive/MyDrive/Dissertation/Datasets/processed/acdc_voc"

### **Delete old registration**

In [12]:
from detectron2.data import DatasetCatalog, MetadataCatalog

for name in [
    "acdc_train",
    "acdc_val",
    "acdc_test",
    "acdc_test_fog",
    "acdc_test_rain",
    "acdc_test_snow"
]:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
    if name in MetadataCatalog.list():
        MetadataCatalog.remove(name)

print("Old ACDC registrations removed.")

Old ACDC registrations removed.


## **Registration**

In [5]:
# Fix numpy compatibility
!pip install -q --force-reinstall numpy==1.26.4

# Build dependencies
!pip install -q setuptools==68.0.0 wheel cython

#Import detectron from the source
%cd /content

import os

if not os.path.exists("/content/detectron2"):
    !git clone https://github.com/facebookresearch/detectron2.git

%cd /content/detectron2
!python -m pip install --no-build-isolation -e .
%cd /content

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3

in case it use the latest version of numpy, restart the runtime

In [ ]:
import os
os.kill(os.getpid(), 9)

In [6]:
import numpy as np
print(np.__version__)

1.26.4


In [7]:
import detectron2
print("Detectron2 imported successfully.")

Detectron2 imported successfully.


In [8]:
import sys

# Force Python to use the real Detectron2 package
sys.path.insert(0, "/content/detectron2")

# Clear wrong cached imports if needed
if "detectron2" in sys.modules:
    del sys.modules["detectron2"]

print(sys.path[:3])

['/content/detectron2', '/content', '/env/python']


In [13]:
from detectron2.data.datasets import register_pascal_voc

ACDC_VOC_ROOT = "/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/acdc_voc"

CLASS_NAMES = ("person", "bicycle", "car", "motorcycle", "bus", "truck")

register_pascal_voc("acdc_train", ACDC_VOC_ROOT, "train", "2024", CLASS_NAMES)
register_pascal_voc("acdc_val", ACDC_VOC_ROOT, "val", "2024", CLASS_NAMES)
register_pascal_voc("acdc_test", ACDC_VOC_ROOT, "test", "2024", CLASS_NAMES)
register_pascal_voc("acdc_test_fog", ACDC_VOC_ROOT, "test_fog", "2024", CLASS_NAMES)
register_pascal_voc("acdc_test_rain", ACDC_VOC_ROOT, "test_rain", "2024", CLASS_NAMES)
register_pascal_voc("acdc_test_snow", ACDC_VOC_ROOT, "test_snow", "2024", CLASS_NAMES)

print("ACDC registered correctly.")

ACDC registered correctly.


### **Mini validation**

In [ ]:
from detectron2.data import DatasetCatalog

for dataset_name in [
    "acdc_train",
    "acdc_val",
    "acdc_test",
    "acdc_test_fog",
    "acdc_test_rain",
    "acdc_test_snow"
]:
    dataset_dicts = DatasetCatalog.get(dataset_name)
    print(dataset_name, ":", len(dataset_dicts), "images")

acdc_train : 1620 images
